# Best-checkpoint audit (no retraining)

This notebook loads `continuous_prefix_lora/checkpoint_best.pt` from Google Drive. It tests correct versus shuffled tables, prints saved-checkpoint predictions, applies WikiTableQuestions complete-denotation scoring, and measures direct gold-answer loss caused by the configured 32×8 truncation.

The default generation audit uses 200 validation examples; truncation coverage always checks the full validation split.

In [ ]:
# Colab: Runtime > Change runtime type > T4 GPU (or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/table-cnn-mrc
%pip install -q -r requirements.txt

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
try:
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
except Exception as error:
    print(f"HF login skipped: {error}")

RUN_DIR = Path("/content/drive/MyDrive/cnn_qwen_table_mcr/outputs/continuous_prefix_lora")
BEST_CHECKPOINT = RUN_DIR / "checkpoint_best.pt"
AUDIT_DIR = RUN_DIR / "diagnostics/checkpoint_audit"
assert BEST_CHECKPOINT.is_file(), f"Missing checkpoint: {BEST_CHECKPOINT}"
print(f"Checkpoint: {BEST_CHECKPOINT}")
print(f"Audit output: {AUDIT_DIR}")

## Run the audit

This cell only performs inference. The first run downloads the official WikiTableQuestions 1.0.2 tagged metadata into Drive; later runs reuse it. Output streams live.

In [ ]:
MAX_EXAMPLES = 200  # use 2831 later for the complete validation generation audit

!PYTHONUNBUFFERED=1 python -u scripts/audit_saved_checkpoint.py \
    --config configs/continuous_prefix_lora.yaml \
    --run-dir "{RUN_DIR}" \
    --checkpoint best \
    --max-examples {MAX_EXAMPLES} \
    --sample-count 5 \
    --output-dir "{AUDIT_DIR}"

In [ ]:
import json
import pandas as pd
from IPython.display import display

with (AUDIT_DIR / "audit_summary.json").open() as handle:
    audit = json.load(handle)

legacy = audit["legacy_any_answer_metrics"]
official = audit["official_denotation_metrics"]
coverage = audit["truncation_coverage"]
display(pd.DataFrame([
    {"metric": "Legacy any-answer EM", "correct table": legacy["correct_table_exact_match"], "shuffled table": legacy["shuffled_table_exact_match"], "drop": legacy["exact_match_drop"]},
    {"metric": "Official denotation accuracy", "correct table": official["correct_table_denotation_accuracy"], "shuffled table": official["shuffled_table_denotation_accuracy"], "drop": official["denotation_accuracy_drop"]},
]))
display(pd.DataFrame([
    {"audit": "Multi-answer examples", "count": coverage["multi_answer_count"], "rate": coverage["multi_answer_rate"]},
    {"audit": "Full-table direct answer coverage", "count": coverage["full_table_direct_coverage_count"], "rate": coverage["full_table_direct_coverage_rate"]},
    {"audit": "Truncated-table direct answer coverage", "count": coverage["truncated_table_direct_coverage_count"], "rate": coverage["truncated_table_direct_coverage_rate"]},
    {"audit": "Complete direct answer removed by truncation", "count": coverage["truncation_removed_direct_answer_count"], "rate": coverage["truncation_removed_rate_overall"]},
]))
print(f"Loaded checkpoint epoch: {audit['checkpoint_epoch']}")
print(f"Prediction change rate after shuffling: {official['prediction_change_rate']:.2%}")

In [ ]:
# Inspect the most informative predictions from the best checkpoint.
for category in [
    "representative",
    "multi_answer",
    "legacy_correct_but_denotation_wrong",
    "prediction_changed_after_shuffle",
    "correct_table_helped",
]:
    rows = audit["samples"][category]
    print(f"\n=== {category} ({len(rows)}) ===")
    if rows:
        display(pd.DataFrame(rows))

print("\n=== examples whose direct answer was removed by truncation ===")
display(pd.DataFrame(coverage["removed_samples"]))

## How to interpret this

- A positive official-accuracy drop after shuffling is evidence that the checkpoint uses table content.
- A large gap between legacy EM and denotation accuracy confirms that single-answer training/evaluation is distorting the result.
- A high truncation-removal rate means the 32×8 crop must be fixed before retraining.
- Direct coverage is deliberately conservative: computed answers may be absent even from the full table and are not counted as truncation failures.

After reviewing this audit, the next code change is to train on a deterministic serialization of every gold answer and use the same complete-denotation metric for validation and early stopping. The existing checkpoint should not be used to test that training fix, because it learned only the first answer.